# Kaggle Qwen3-8B 4-bit Mode Prediction

Use this notebook after you already have the Llama/Ollama + human verification dataset. It adds a `Qwen3-8B 4-bit` prediction column and a `mode_prediction` column to the same 150-row CSV.

## 1. Install Ollama

In [ ]:
!apt-get update -qq
!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

## 2. Start Ollama

In [ ]:
import subprocess
import time

ollama_server = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

time.sleep(10)
print("Ollama server started")

## 3. Pull Qwen3-8B

In [ ]:
MODEL_NAME = "qwen3:8b"
QWEN_COLUMN = "Qwen3-8B 4-bit"

!ollama pull qwen3:8b
!ollama list

## 4. Clone ResearchLanka AI Repo

In [ ]:
from pathlib import Path
import shutil

repo_dir = Path("/kaggle/working/researchlanka-ai")

%cd /kaggle/working

if repo_dir.exists():
    shutil.rmtree(repo_dir)

!git clone --branch feature/ai-paper-filtering https://github.com/krish-anu/researchlanka-ai.git /kaggle/working/researchlanka-ai

%cd /kaggle/working/researchlanka-ai/backend
!pip install -q -r requirements.txt

## 5. Copy Your Human Verification Dataset

Add the CSV dataset in the Kaggle right sidebar first. This cell searches `/kaggle/input` for the completed human verification file.

In [ ]:
import pandas as pd

input_matches = sorted(Path("/kaggle/input").rglob("*human*verification*.csv"))
print("Found candidate files:")
for match in input_matches:
    print(" -", match)

if not input_matches:
    raise FileNotFoundError("Could not find a human verification CSV under /kaggle/input.")

src = input_matches[0]
dst = Path("data/processed/ai/ai_llm_150_human_verification_ollama_kaggle.csv")
dst.parent.mkdir(parents=True, exist_ok=True)
shutil.copy(src, dst)

df = pd.read_csv(dst)
print("Copied:", src, "->", dst)
print("shape:", df.shape)
print("columns:", list(df.columns))
df.head()

## 6. Quick Qwen Test

In [ ]:
!ollama run qwen3:8b "Return one word only: READY"

## 7. Add Qwen3-8B 4-bit Predictions

This writes after every row, so you can rerun the cell to resume if Kaggle stops.

In [ ]:
import json
import os
import sys
from IPython.display import clear_output

sys.path.insert(0, str(Path.cwd()))

from src.ai_relevance.config import GeminiConfig
from src.ai_relevance.fields import publication_metadata
from src.ai_relevance.gemini_client import OllamaAIClient

INPUT_PATH = Path("data/processed/ai/ai_llm_150_human_verification_ollama_kaggle.csv")
OUTPUT_PATH = Path("data/processed/ai/ai_llm_150_human_verification_qwen3_mode_prediction.csv")

df = pd.read_csv(INPUT_PATH)
if OUTPUT_PATH.exists():
    saved = pd.read_csv(OUTPUT_PATH)
    if len(saved) == len(df):
        df = saved
        print("Resuming from", OUTPUT_PATH)

new_columns = {
    QWEN_COLUMN: "",
    "qwen3_confidence": pd.NA,
    "qwen3_category": "",
    "qwen3_reason": "",
    "qwen3_evidence": "",
    "qwen3_status": "",
    "qwen3_error": "",
}
for column, default in new_columns.items():
    if column not in df.columns:
        df[column] = default

config = GeminiConfig(
    provider="ollama",
    model=MODEL_NAME,
    prompt_version="v3",
    timeout_seconds=300,
    max_retries=1,
    ollama_seed=42,
)
client = OllamaAIClient(config)

pending = df[~df["qwen3_status"].eq("success")].index.tolist()
print(f"Pending rows: {len(pending)} / {len(df)}")

for position, idx in enumerate(pending, start=1):
    row = df.loc[idx]
    try:
        result = client.classify(publication_metadata(row, fallback=idx))
        classification = result.classification
        df.at[idx, QWEN_COLUMN] = classification.label
        df.at[idx, "qwen3_confidence"] = classification.confidence
        df.at[idx, "qwen3_category"] = classification.ai_category
        df.at[idx, "qwen3_reason"] = classification.reason
        df.at[idx, "qwen3_evidence"] = " | ".join(classification.evidence)
        df.at[idx, "qwen3_status"] = "success"
        df.at[idx, "qwen3_error"] = ""
    except Exception as exc:
        df.at[idx, "qwen3_status"] = "failed"
        df.at[idx, "qwen3_error"] = str(exc)

    df.to_csv(OUTPUT_PATH, index=False)
    clear_output(wait=True)
    print(f"Processed {position} / {len(pending)} pending rows")
    print(df["qwen3_status"].value_counts(dropna=False))
    print(df[QWEN_COLUMN].value_counts(dropna=False))

print("Saved:", OUTPUT_PATH)

## 8. Create Mode Prediction Column

In [ ]:
from collections import Counter

def normalize_label(value):
    if pd.isna(value):
        return None
    text = str(value).strip().upper()
    if text in {"TRUE", "AI", "YES", "1"}:
        return "AI"
    if text in {"FALSE", "NON_AI", "NON-AI", "NO", "0"}:
        return "NON_AI"
    if text == "REVIEW":
        return "REVIEW"
    return None

def mode_prediction(row):
    labels = [
        normalize_label(row.get("ai_llm_label")),
        normalize_label(row.get("human_label")),
        normalize_label(row.get(QWEN_COLUMN)),
    ]
    labels = [label for label in labels if label is not None]
    if not labels:
        return "REVIEW"
    counts = Counter(labels)
    top_count = counts.most_common(1)[0][1]
    winners = sorted(label for label, count in counts.items() if count == top_count)
    if len(winners) == 1:
        return winners[0]
    human_label = normalize_label(row.get("human_label"))
    return human_label if human_label in winners else "REVIEW"

df = pd.read_csv(OUTPUT_PATH)
df["mode_prediction"] = df.apply(mode_prediction, axis=1)
df.to_csv(OUTPUT_PATH, index=False)

print("Saved:", OUTPUT_PATH)
print("Qwen predictions:")
print(df[QWEN_COLUMN].value_counts(dropna=False))
print("Mode predictions:")
print(df["mode_prediction"].value_counts(dropna=False))
df[["publication_id", "ai_llm_label", "human_label", QWEN_COLUMN, "mode_prediction", "qwen3_confidence", "qwen3_reason"]].head(20)

## 9. Zip Output For Download

In [ ]:
!zip -j /kaggle/working/qwen3_mode_prediction_output.zip data/processed/ai/ai_llm_150_human_verification_qwen3_mode_prediction.csv
print("Download: /kaggle/working/qwen3_mode_prediction_output.zip")